# Spring v2 — Zero-Shot Video Anomaly Detection

| | |
|---|---|
| **Backbone** | CLIP ViT-B/32 (same as v1) |
| **Prompts** | Same as v1 |
| **Aggregation** | **Top-K Mean** — mean of top 10% segment anomaly scores |
| **Threshold** | Grid-searched on validation set (maximise balanced accuracy) |
| **Goal** | Analyse effect of aggregation strategy vs Spring v1 (max score) |

---

### What changed from v1
| | Spring v1 | Spring v2 |
|---|---|---|
| Backbone | ViT-B/32 | ViT-B/32 (unchanged) |
| Prompts | Simple 5+4 set | Same (unchanged) |
| Video aggregation | `max(scores)` | `mean(top 10% scores)` |

> Edit **Cell 1 (Configuration)** if your paths differ from v1.

## 0 — Install dependencies
Run once. Skip if already installed.

In [ ]:
import sys
!{sys.executable} -m pip install -q openai-clip torch torchvision scikit-learn matplotlib pandas pillow

## 1 — Configuration
**Only `RESULTS_DIR` changed from v1. Everything else is identical.**

In [ ]:
import os, glob as _glob

# ── Auto-detect manifest path ─────────────────────────────────────────────
_search_roots = ['/', '/home', '/root', '/content', os.getcwd()]
_manifest_found = None
for _root in _search_roots:
    for _dirpath, _dirs, _files in os.walk(_root):
        _dirs[:] = [d for d in _dirs if d not in
                    {'proc','sys','dev','run','snap','boot','lib','lib64','usr','bin','sbin','etc'}]
        if 'manifest.csv' in _files:
            _manifest_found = os.path.join(_dirpath, 'manifest.csv')
            break
    if _manifest_found:
        break

MANIFEST_PATH = _manifest_found if _manifest_found else '/data/manifests_new/manifest.csv'
SEGMENT_ROOT  = os.path.dirname(os.path.dirname(MANIFEST_PATH)).replace('manifests_new','segments_new') if _manifest_found else '/data/segments_new'
RESULTS_DIR   = 'spring_v2_results'          # ← v2 output folder

# ── Top-K parameter ───────────────────────────────────────────────────────
TOPK_RATIO = 0.10    # use top 10% of segments per video

print(f'MANIFEST_PATH : {MANIFEST_PATH}')
print(f'  exists       : {os.path.exists(MANIFEST_PATH)}')
print(f'SEGMENT_ROOT  : {SEGMENT_ROOT}')
print(f'  exists       : {os.path.exists(SEGMENT_ROOT)}')
print(f'TOPK_RATIO    : {TOPK_RATIO}')

# ── Column map (same as v1) ────────────────────────────────────────────────
COLUMN_MAP = {
    'video_id' : 'video_id',
    'label'    : 'class',
    'split'    : 'split',
    'path'     : 'path',
}

# ── Prompts (identical to v1) ─────────────────────────────────────────────
ANOMALY_PROMPTS = [
    'a robbery happening',
    'a person fighting',
    'a violent action',
    'people fighting or attacking each other',
    'criminal activity in a surveillance video',
]
NORMAL_PROMPTS = [
    'people walking normally',
    'a peaceful street scene',
    'normal activity in a public place',
    'ordinary daily life',
]

print('\nConfiguration loaded.')

## 2 — Imports & Device

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
from PIL import Image
import torch
import clip
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, balanced_accuracy_score
)
import matplotlib.pyplot as plt
%matplotlib inline
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Device  : {DEVICE}')
print(f'Results : ./{RESULTS_DIR}/')

## 3 — Load Manifest

In [ ]:
try:
    raw = pd.read_csv(MANIFEST_PATH, encoding='utf-8')
except UnicodeDecodeError:
    raw = pd.read_csv(MANIFEST_PATH, encoding='latin-1')

print(f'Columns : {raw.columns.tolist()}')
print(f'Shape   : {raw.shape}')
raw.head(3)

In [ ]:
# Fix duplicate columns
if raw.columns.duplicated().any():
    raw = raw.loc[:, ~raw.columns.duplicated()]

df = raw.copy()

# 'class' = category name, 'label' = 0/1 binary
if 'class' in df.columns:
    df = df.rename(columns={'class': 'label_name'})
if 'label' in df.columns:
    df = df.rename(columns={'label': 'is_anomaly_raw'})
if 'label_name' in df.columns:
    df['label'] = df['label_name'].astype(str).str.strip()
elif 'is_anomaly_raw' in df.columns:
    df['label'] = df['is_anomaly_raw'].apply(lambda x: 'Anomaly' if int(x)==1 else 'Normal')

df['split'] = df['split'].astype(str).str.strip().str.lower().replace({'val': 'validation'})

missing = {'video_id','label','split','path'} - set(df.columns)
if missing:
    raise ValueError(f'Missing columns: {missing} | Available: {df.columns.tolist()}')

print('Segments per split:')
print(df['split'].value_counts().to_string())
print('\nSegments per label:')
print(df['label'].value_counts().to_string())
print(f'\nTotal: {len(df)}')

## 4 — Load CLIP ViT-B/32 & Encode Prompts
Identical to Spring v1.

In [ ]:
print('Loading CLIP ViT-B/32 …')
model, preprocess = clip.load('ViT-B/32', device=DEVICE)
model.eval()
print('Done.')

In [ ]:
def encode_prompts(model, prompts):
    tokens = clip.tokenize(prompts).to(DEVICE)
    with torch.no_grad():
        embs = model.encode_text(tokens).float()
    embs = embs / embs.norm(dim=-1, keepdim=True)
    mean = embs.mean(0)
    return mean / mean.norm()

anom_emb = encode_prompts(model, ANOMALY_PROMPTS)
norm_emb = encode_prompts(model, NORMAL_PROMPTS)
print(f'Anomaly prompts : {len(ANOMALY_PROMPTS)}')
print(f'Normal  prompts : {len(NORMAL_PROMPTS)}')

## 5 — Segment-Level Inference
Identical to Spring v1 — per-segment cosine similarity scores.
The difference comes in the **next cell** (aggregation).

In [ ]:
def resolve_path(raw_path):
    if os.path.isabs(raw_path) and os.path.exists(raw_path):
        return raw_path
    joined = os.path.join(SEGMENT_ROOT, raw_path)
    if os.path.exists(joined):
        return joined
    basename = os.path.basename(raw_path.rstrip('/\\'))
    for root, dirs, _ in os.walk(SEGMENT_ROOT):
        if basename in dirs:
            return os.path.join(root, basename)
    return joined

def load_frames(seg_path, max_frames=16):
    full = resolve_path(seg_path)
    if os.path.isdir(full):
        frames = []
        for ext in ('*.jpg','*.jpeg','*.png'):
            frames.extend(sorted(glob.glob(os.path.join(full, ext))))
        frames = frames[:max_frames]
        if not frames:
            return None
        return [preprocess(Image.open(f).convert('RGB')) for f in frames]
    if os.path.isfile(full):
        try:
            return [preprocess(Image.open(full).convert('RGB'))]
        except Exception:
            return None
    return None

def segment_score(frames):
    batch = torch.stack(frames).to(DEVICE)
    with torch.no_grad():
        img_embs = model.encode_image(batch).float()
    img_embs = img_embs / img_embs.norm(dim=-1, keepdim=True)
    seg_emb  = img_embs.mean(0)
    seg_emb  = seg_emb / seg_emb.norm()
    return (seg_emb @ anom_emb).item(), (seg_emb @ norm_emb).item()

print('Helper functions defined.')

In [ ]:
records = []
n = len(df)

for i, (_, row) in enumerate(df.iterrows()):
    frames = load_frames(row['path'])
    if frames is None:
        print(f'  [WARN] Cannot load: {row["path"]}')
        sa, sn = np.nan, np.nan
    else:
        sa, sn = segment_score(frames)
    records.append({
        'video_id'     : row['video_id'],
        'label'        : row['label'],
        'split'        : row['split'],
        'score_anomaly': sa,
        'score_normal' : sn,
        'score_diff'   : (sa - sn) if not np.isnan(sa) else np.nan,
    })
    if (i + 1) % 50 == 0 or (i + 1) == n:
        print(f'  [{i+1}/{n}] processed')

seg_df = pd.DataFrame(records)
seg_df.to_csv(os.path.join(RESULTS_DIR, 'segment_scores.csv'), index=False)
print(f'\nSegment scores saved. Shape: {seg_df.shape}')
seg_df.head()

## 6 — Video-Level Aggregation: Top-K Mean

**Spring v2 change:** instead of `max(scores)`, we take the mean of the
top `TOPK_RATIO` (10%) highest anomaly scores per video.

```
video_score = mean( sort_desc(segment_scores)[ : ceil(N × 0.10) ] )
```

This reduces the influence of a single noisy high-scoring segment
while still emphasising the most suspicious temporal regions.

In [ ]:
import math

seg_df['video_id'] = seg_df['video_id'].astype(str).str.strip()
seg_df['split']    = seg_df['split'].astype(str).str.strip()
seg_df['label']    = seg_df['label'].astype(str).str.strip()

if 'is_anomaly_raw' in df.columns:
    id_map = df.drop_duplicates('video_id').set_index('video_id')['is_anomaly_raw'].to_dict()
else:
    id_map = None

rows = []
for (vid, split), grp in seg_df.groupby(['video_id', 'split']):
    grp = grp.dropna(subset=['score_anomaly'])
    if grp.empty:
        continue
    label = grp['label'].iloc[0]

    # ── Top-K mean aggregation ─────────────────────────────────────────────
    scores_sorted = grp['score_anomaly'].sort_values(ascending=False).values
    k = max(1, math.ceil(len(scores_sorted) * TOPK_RATIO))   # at least 1
    topk_score = scores_sorted[:k].mean()                    # mean of top-k

    if id_map is not None and vid in id_map:
        is_anom = int(id_map[vid])
    else:
        is_anom = int(label.lower() not in ('normal','normalvideos','normal videos'))

    rows.append({
        'video_id'   : vid,
        'split'      : split,
        'label'      : label,
        'is_anomaly' : is_anom,
        'video_score': topk_score,
        'n_segments' : len(grp),
        'k_used'     : k,
    })

video_df = pd.DataFrame(rows)
video_df.to_csv(os.path.join(RESULTS_DIR, 'video_scores.csv'), index=False)

print(f'Aggregation: Top-{TOPK_RATIO*100:.0f}% mean')
print(f'Avg k per video: {video_df["k_used"].mean():.1f} segments')
print('\nVideos per split and class:')
print(video_df.groupby(['split','is_anomaly']).size().to_string())
video_df.head()

## 7 — Threshold Optimisation (Validation Set)
Same strategy as v1: grid-search maximising balanced accuracy.

In [ ]:
val = video_df[video_df['split'] == 'validation']
if val.empty:
    raise ValueError("No validation rows found. Check manifest 'split' values.")

y_val  = val['is_anomaly'].values
sc_val = val['video_score'].values

best_t, best_ba = 0.0, -1.0
thresholds, bal_accs = [], []

for t in np.linspace(sc_val.min(), sc_val.max(), 300):
    ba = balanced_accuracy_score(y_val, (sc_val >= t).astype(int))
    thresholds.append(t)
    bal_accs.append(ba)
    if ba > best_ba:
        best_ba, best_t = ba, t

print(f'Best threshold  τ  = {best_t:.4f}')
print(f'Val Balanced Acc   = {best_ba:.4f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(thresholds, bal_accs, color='steelblue', linewidth=1.5)
ax.axvline(best_t, color='red', linestyle='--', label=f'τ = {best_t:.4f}')
ax.set(xlabel='Threshold', ylabel='Balanced Accuracy',
       title='Threshold Sweep — Validation Set (Spring v2, Top-K Mean)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'threshold_sweep.png'), dpi=150)
plt.show()

## 8 — Evaluation

In [ ]:
def evaluate_split(video_df, threshold, split):
    sub = video_df[video_df['split'] == split]
    if sub.empty:
        print(f"No rows for split='{split}'")
        return {}
    y    = sub['is_anomaly'].values
    pred = (sub['video_score'].values >= threshold).astype(int)
    prec = precision_score(y, pred, zero_division=0)
    rec  = recall_score(y, pred, zero_division=0)
    f1   = f1_score(y, pred, zero_division=0)
    bac  = balanced_accuracy_score(y, pred)
    cm   = confusion_matrix(y, pred)
    try:
        auc = roc_auc_score(y, sub['video_score'].values)
    except Exception:
        auc = float('nan')
    tn, fp, fn, tp = cm.ravel() if cm.shape==(2,2) else (0,0,0,0)
    return dict(split=split, threshold=threshold,
                precision=prec, recall=rec, f1=f1,
                balanced_accuracy=bac, roc_auc=auc,
                TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn))

print('evaluate_split() defined.')

In [ ]:
val_metrics = evaluate_split(video_df, best_t, 'validation')
print('VALIDATION SET')
for k,v in val_metrics.items():
    if k not in ('split',):
        print(f'  {k:<22}: {v:.4f}' if isinstance(v, float) else f'  {k:<22}: {v}')

In [ ]:
test_metrics = evaluate_split(video_df, best_t, 'test')
print('TEST SET')
for k,v in test_metrics.items():
    if k not in ('split',):
        print(f'  {k:<22}: {v:.4f}' if isinstance(v, float) else f'  {k:<22}: {v}')

## 9 — Plots

In [ ]:
def plot_confusion_matrix(m, split):
    arr = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(arr, cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set(xticks=[0,1], yticks=[0,1],
           xticklabels=['Normal','Anomaly'],
           yticklabels=['Normal','Anomaly'],
           xlabel='Predicted', ylabel='Actual',
           title=f'Confusion Matrix — {split.capitalize()}\n'
                 f'(Spring v2, ViT-B/32, Top-K Mean)')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(arr[i,j]), ha='center', va='center',
                    fontsize=14, fontweight='bold',
                    color='white' if arr[i,j] > arr.max()/2 else 'black')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'confusion_matrix_{split}.png'), dpi=150)
    plt.show()

plot_confusion_matrix(val_metrics, 'validation')
plot_confusion_matrix(test_metrics, 'test')

In [ ]:
def plot_score_distribution(video_df, threshold, split):
    sub  = video_df[video_df['split'] == split]
    norm = sub[sub['is_anomaly']==0]['video_score']
    anom = sub[sub['is_anomaly']==1]['video_score']
    bins = np.linspace(sub['video_score'].min()-0.002,
                       sub['video_score'].max()+0.002, 30)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(norm, bins=bins, alpha=0.6, label='Normal',  color='steelblue')
    ax.hist(anom, bins=bins, alpha=0.6, label='Anomaly', color='darkorange')
    ax.axvline(threshold, color='red', linestyle='--',
               linewidth=1.5, label=f'τ = {threshold:.4f}')
    ax.set(xlabel='Top-K Mean Anomaly Score', ylabel='Count',
           title=f'Score Distribution — {split.capitalize()} (Spring v2)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'score_distribution_{split}.png'), dpi=150)
    plt.show()

plot_score_distribution(video_df, best_t, 'validation')
plot_score_distribution(video_df, best_t, 'test')

In [ ]:
results = [val_metrics, test_metrics]
res_df  = pd.DataFrame(results)[['split','precision','recall','f1']]
x = np.arange(len(res_df)); w = 0.25
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x-w, res_df['precision'], w, label='Precision', color='steelblue')
ax.bar(x,   res_df['recall'],   w, label='Recall',    color='darkorange')
ax.bar(x+w, res_df['f1'],       w, label='F1',        color='seagreen')
ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in res_df['split']])
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title('Spring v2 — Performance Summary (ViT-B/32, Top-K Mean)')
ax.legend()
for bar in ax.patches:
    h = bar.get_height()
    if h > 0.01:
        ax.text(bar.get_x()+bar.get_width()/2, h+0.01,
                f'{h:.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'performance_summary.png'), dpi=150)
plt.show()

## 10 — v1 vs v2 Comparison

Load v1 results (if available) and plot side-by-side.
> Skip this cell if you haven't run the v1 notebook yet.

In [ ]:
V1_METRICS_CSV = 'spring_v1_results/metrics_summary.csv'

if os.path.exists(V1_METRICS_CSV):
    v1_df = pd.read_csv(V1_METRICS_CSV)
    v1_test = v1_df[v1_df['split']=='test'].iloc[0]
    v2_test = test_metrics

    compare_df = pd.DataFrame([
        {'Version':'Spring v1 (max)',    'Precision':v1_test['precision'], 'Recall':v1_test['recall'], 'F1':v1_test['f1'],
         'Balanced Acc':v1_test['balanced_accuracy'], 'ROC-AUC':v1_test['roc_auc']},
        {'Version':'Spring v2 (top-k)', 'Precision':v2_test['precision'], 'Recall':v2_test['recall'], 'F1':v2_test['f1'],
         'Balanced Acc':v2_test['balanced_accuracy'], 'ROC-AUC':v2_test['roc_auc']},
    ])

    print('=' * 60)
    print('  AGGREGATION STRATEGY COMPARISON — TEST SET')
    print('=' * 60)
    print(compare_df.to_string(index=False))
    print('=' * 60)

    # Delta row
    for col in ['Precision','Recall','F1','Balanced Acc','ROC-AUC']:
        delta = compare_df.loc[1,col] - compare_df.loc[0,col]
        sign  = '+' if delta >= 0 else ''
        print(f'  Δ {col:<15}: {sign}{delta:.4f}')

    # Bar chart
    metrics_to_plot = ['Precision','Recall','F1','Balanced Acc']
    x = np.arange(len(metrics_to_plot)); w = 0.35
    v1_vals = [compare_df.loc[0, m] for m in metrics_to_plot]
    v2_vals = [compare_df.loc[1, m] for m in metrics_to_plot]

    fig, ax = plt.subplots(figsize=(9, 4))
    bars1 = ax.bar(x-w/2, v1_vals, w, label='v1 (max score)',    color='steelblue',   alpha=0.85)
    bars2 = ax.bar(x+w/2, v2_vals, w, label='v2 (top-k mean)',   color='darkorange',  alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_to_plot)
    ax.set_ylim(0, 1.2); ax.set_ylabel('Score')
    ax.set_title('Spring v1 vs v2 — Test Set Comparison\n(ViT-B/32, Max Score vs Top-K Mean)')
    ax.legend()
    for bar in list(bars1) + list(bars2):
        h = bar.get_height()
        if h > 0.01:
            ax.text(bar.get_x()+bar.get_width()/2, h+0.01,
                    f'{h:.2f}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'v1_vs_v2_comparison.png'), dpi=150)
    plt.show()
    compare_df.to_csv(os.path.join(RESULTS_DIR, 'v1_vs_v2_comparison.csv'), index=False)
else:
    print(f'v1 results not found at {V1_METRICS_CSV}')
    print('Run the Spring v1 notebook first, then re-run this cell.')

## 11 — Save Results & Print Table 10 Row

In [ ]:
metrics_csv = os.path.join(RESULTS_DIR, 'metrics_summary.csv')
pd.DataFrame([val_metrics, test_metrics]).to_csv(metrics_csv, index=False)
print(f'Metrics saved → {metrics_csv}')

tm = test_metrics
print('\n' + '='*56)
print('  SPRING v2 — FINAL TEST RESULTS')
print('='*56)
print(f'  Backbone    : ViT-B/32')
print(f'  Aggregation : Top-{TOPK_RATIO*100:.0f}% segment mean score')
print(f'  Threshold   : {best_t:.4f}')
print(f'  Precision   : {tm["precision"]:.4f}')
print(f'  Recall      : {tm["recall"]:.4f}')
print(f'  F1-Score    : {tm["f1"]:.4f}')
print(f'  ROC-AUC     : {tm["roc_auc"]:.4f}')
print(f'  TP={tm["TP"]}  FP={tm["FP"]}  FN={tm["FN"]}  TN={tm["TN"]}')
print('='*56)
print(f'\n  → Table 10 row:')
print(f'  Spring v2 | ViT-B/32, Top-K mean | '
      f'{tm["precision"]:.2f} | {tm["recall"]:.2f} | {tm["f1"]:.2f}')